# 巨大数ポーカー AI — Colab で学習する

**スマホでファイルをダウンロードして Drive に移す必要はありません。**
Colab から GitHub を直接 clone できます。

手順は上から順にセルを実行するだけ。

1. Drive をマウント（学習ログを永続化するため）
2. GitHub から clone
3. Node.js を確認（巨大数エンジンは JS 側が持っている）
4. 学習
5. 結果を Drive に保存 → 手元の GUI で見る

> ランタイムは「GPU」にしておくと少し速くなりますが、
> この学習はネットワークが小さく **環境（Node）側が律速** なので、
> CPU ランタイムでもほとんど変わりません。

In [ ]:
# ── 1. Drive をマウント ──────────────────────────────
# ここに学習ログを写しておくと、ランタイムが切れてもグラフのデータが残る。
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/huge_number_poker_checkpoints'
os.makedirs(DATA_DIR, exist_ok=True)
print('永続フォルダ:', DATA_DIR)

In [ ]:
# ── 2. GitHub から取得 ───────────────────────────────
# ▼ 自分のリポジトリURLに書き換える
REPO = 'https://github.com/<あなたのユーザー名>/Huge_Number_Poker.git'

%cd /content
![ -d Huge_Number_Poker ] && (cd Huge_Number_Poker && git pull) || git clone $REPO
%cd /content/Huge_Number_Poker
!ls

In [ ]:
# ── 3. Node.js の確認 ────────────────────────────────
# 環境（ゲームのルールと巨大数エンジン）は Node 側にある。
# Colab には最初から入っているが、無ければここで入れる。
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs)
!node --version

# 環境が単体で動くか先に確かめる（ここで落ちたら学習しても無駄）
!node train/env_server.js --selfplay 300 --level skilled

In [ ]:
# ── 4. 学習 ──────────────────────────────────────────
# 世代数・並列環境数は環境変数で渡す（GUI から起動するときと同じ経路）。
import os
os.environ['HNP_MAX_GENS']  = '100'    # 何世代回すか
os.environ['HNP_NUM_ENVS']  = '256'    # 並列する卓の数
os.environ['HNP_DECISIONS'] = '8192'   # 1世代で集める意思決定の数
os.environ['HNP_LEVEL']     = 'skilled'  # 相手にする「人間」の計算力
os.environ['HNP_EVAL_EVERY'] = '5'

!python train/train.py

In [ ]:
# ── 5. 結果を Drive へ ───────────────────────────────
# train.py は学習ログを自動で Drive に写すが、
# 重み（policy_*.json）と チェックポイント も置いておく。
import shutil, glob, os
for src in glob.glob('models/policy_*.json') + \
           glob.glob('train/models/*_log.json') + \
           glob.glob('train/models/*_best.pt'):
    dst = os.path.join(DATA_DIR, os.path.basename(src))
    shutil.copy2(src, dst)
    print('→', dst)

## 学習の様子を見るには

**おすすめ: Colab で回して、手元の GUI で見る。**

上の 5. で `ppo_<レベル>_log.json` が Drive に入るので、
手元の PC にダウンロードして `train/models/` に置き、

```bash
python train/launcher.py
```

を起動すればグラフと表で見られます。

---

**Colab の中で GUI を開きたい場合**は次のセル。
ただし Colab のプロキシ越しなので、SSE（リアルタイム更新）が
詰まることがあります。学習の進行は上のセルの `[Progress]` 行でも追えます。

In [ ]:
# ── （任意）Colab 内でダッシュボードを開く ─────────────
from google.colab import output
import threading, sys
sys.path.insert(0, '/content/Huge_Number_Poker/train')
from dashboard_server import app, PORT

threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=PORT, debug=False, threaded=True),
    daemon=True).start()
output.serve_kernel_port_as_window(PORT)

## 学習した AI と対戦する

`models/policy_<レベル>.json` がゲーム本体から読まれる重みです。
Drive から手元にダウンロードして `models/` に置き、

```bash
npm start
```

→ 「1端末で遊ぶ」→ CPU の強さを学習したレベルに合わせる。
重みが読めていれば、そのレベルの CPU が学習済みの打ち方をします
（読めなければヒューリスティック方策で動くので、無くてもゲームは成立します）。

> **注意:** 特徴量（`js/ai-policy.js` の `OBS_DIM`）を変えたら、
> 重みは互換性を失います。読み込み時に次元を照合して、
> 合わなければ警告を出して無視するようにしてあります。